# Spark Tune - Databricks ML Pipeline Demo

End-to-end ML pipeline reading data from a Databricks catalog and demonstrating:

1. **ydata-profiling** - Data exploration & quality profiling
2. **featuretools** - Automated feature generation via Deep Feature Synthesis
3. **tsfresh** - Time-series feature extraction & exploration
4. **XGBoost** - Model training with SparkXGBoost
5. **SHAP** - Model explainability with Shapley values
6. **Insight Analyzer** - Microsegment discovery & lift vs support analysis

In [0]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

In [0]:
# Uncomment to install dependencies if needed
# !pip install -r /Workspace/Users/yadvendra@aidetic.in/spark_beyond/requirements.txt

---
## 1. Load Data from Databricks Catalog

In [0]:
from backend.core.utils import process_col_names

# Read from Databricks Unity Catalog volume
# df = spark.read.csv(
#     "/Volumes/aidetic_databricks/default/credit_card_transactions/credit_card_transactions.csv",
#     header=True,
#     inferSchema=True
# )

# df = df.drop("Unnamed: 0")

CATALOG_NAME = "aidetic_databricks"
SCHEMA_NAME = "default"

credit_card_transactions_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.credit_card_transactions"

df = spark.read.table(credit_card_transactions_table_name)

print(f"Dataset shape: {df.count():,} rows x {len(df.columns)} columns")
df.printSchema()

In [0]:
display(df.limit(5))

---
## 2. Problem Definition & Schema Validation

In [0]:
from backend.core.discovery import Problem, SchemaChecks

problem = Problem(
    target="is_fraud",
    type="classification",
    desired_result=1,
    date_column="trans_date_trans_time"
)

schema_checker = SchemaChecks(dataframe=df, problem=problem)
schema_info = schema_checker.check()

print(f"Problem Type: {problem.type}")
print(f"Target Column: {problem.target}")
print(f"Desired Result: {problem.desired_result}")
print(f"\nSchema Summary:")
print(f"  Categorical columns: {len(schema_info['categorical'])}")
print(f"  Numerical columns: {len(schema_info['numerical'])}")
print(f"  Boolean columns: {len(schema_info['boolean'])}")

---
## 3. YData Profiling - Data Exploration

Generate a comprehensive data profile using ydata-profiling.
The `quick_profile` function auto-samples large datasets to avoid memory issues.

In [0]:
from ydata_profiling import ProfileReport

# Full profiling report (saved to HTML)
profile = ProfileReport(
    df.sample(fraction=0.01, seed=42).toPandas(),
    title="Credit Card Transactions - Profiling Report",
    explorative=True
)

profile.to_file("data_profiling_report.html")
print("Full profiling report saved to 'data_profiling_report.html'")

In [0]:
from backend.core.profiling.ydata_profiler import quick_profile

# Quick profile for summary stats & alerts
quick_stats = quick_profile(df, max_rows=10000)

summary = quick_stats['summary']
print("QUICK PROFILE SUMMARY:")
print(f"  Rows: {summary.get('n_rows', 0):,}")
print(f"  Columns: {summary.get('n_columns', 0)}")
print(f"  Missing Cells: {summary.get('missing_cells_pct', 0):.2f}%")
print(f"  Duplicate Rows: {summary.get('duplicate_rows_pct', 0):.2f}%")

In [0]:
# Display alerts from profiling
print("DATA ALERTS:")
print("-" * 40)
if quick_stats['alerts']:
    for alert in quick_stats['alerts'][:15]:
        print(f"  - {alert['column']}: {alert['type']}")
else:
    print("  No alerts detected.")

print("\nPROFILING RECOMMENDATIONS:")
print("-" * 40)
if quick_stats['recommendations']:
    for rec in quick_stats['recommendations'][:10]:
        print(f"  [{rec['priority'].upper()}] {rec['column']}: {rec['action']}")
else:
    print("  No recommendations.")

---
## 4. Time-Series Detection

Automatically identify temporal structure in the dataset before running tsfresh.

In [0]:
from backend.core.utils.time_series_detector import detect_time_series_structure

ts_info = detect_time_series_structure(df, schema_checker)

print("TIME SERIES DETECTION RESULTS:")
print("-" * 40)
print(f"  Is Time Series: {ts_info.is_time_series}")
print(f"  Time Column: {ts_info.time_column or 'N/A'}")
print(f"  Frequency: {ts_info.frequency.value if ts_info.frequency else 'N/A'}")
print(f"  Entity Columns: {ts_info.entity_columns or 'N/A'}")

if ts_info.recommended_features:
    print("\nRecommended Time-Series Features:")
    for feature in ts_info.recommended_features:
        print(f"    - {feature}")

---
## 5. Featuretools - Automated Feature Generation

Use Deep Feature Synthesis (DFS) to automatically discover and generate
features from the transaction data. Featuretools creates transform and
aggregation features based on the entity relationships.

In [0]:
# !pip install featuretools

In [0]:
from backend.core.features.featuretools_engine import FeaturetoolsEngine

ft_engine = FeaturetoolsEngine(
    spark=spark,
    max_rows_for_pandas=50000,
    verbose=True
)

# Run DFS on a single table
# Use trans_num as the unique index column
ft_result = ft_engine.run_dfs_single_table(
    spark_df=df,
    index_col="trans_num",
    target_entity="transactions",
    time_index="trans_date_trans_time",
    max_depth=1,
    trans_primitives=["add_numeric", "subtract_numeric", "multiply_numeric"],
    max_features=50
)

print(f"\nFEATURETOOLS RESULTS:")
print(f"  Features generated: {len(ft_result.feature_names)}")
print(f"  Generation time: {ft_result.generation_time:.2f}s")
print(f"\nSample generated feature names:")
for name in ft_result.feature_names[:10]:
    print(f"    - {name}")

# Merge featuretools features back into the main DataFrame
df = ft_engine.to_spark(ft_result.feature_matrix, df, join_column="trans_num")

print(f"\nDataFrame after featuretools merge: {len(df.columns)} columns")

In [0]:
# Get human-readable feature descriptions
descriptions = ft_engine.get_feature_descriptions()

print("FEATURE DESCRIPTIONS (first 10):")
print("-" * 60)
for desc in descriptions[:10]:
    print(f"  {desc['name']}: {desc['type']}")

In [0]:
# Preview the feature matrix
ft_result.feature_matrix.head()

---
## 6. TSFresh - Time-Series Feature Extraction

Extract time-series features per entity (e.g., per credit card) using tsfresh.
These are entity-level aggregates (one row per `cc_num`) that get joined back
to the transaction-level DataFrame so each transaction inherits its entity's
time-series statistics.

Supports three extraction modes:
- **minimal** (~10 features per column)
- **efficient** (~100 features per column)
- **comprehensive** (~750 features per column)

In [0]:
# !pip install tsfresh

In [0]:
from backend.core.features.tsfresh_engine import TSFreshEngine, FeatureExtractionMode

# Fix: Remove duplicate columns by renaming all columns to unique names first
col_names = df.columns
seen = {}
unique_names = []
for col in col_names:
    if col not in seen:
        seen[col] = 0
        unique_names.append(col)
    else:
        seen[col] += 1
        unique_names.append(f"{col}_dup{seen[col]}")

# Rename all columns to unique names
df = df.toDF(*unique_names)

# Now select only the original columns (without _dup suffix)
original_cols = [col for col in unique_names if not col.endswith(tuple(f"_dup{i}" for i in range(1, 100)))]
df = df.select(original_cols)

ts_engine = TSFreshEngine(
    spark=spark,
    max_rows_for_pandas=50000,
    n_jobs=0,  # use all cores
    verbose=True
)

# Extract time-series features in minimal mode for speed
ts_result = ts_engine.extract_features(
    spark_df=df,
    id_column="cc_num",                     # entity: credit card number
    time_column="trans_date_trans_time",     # temporal column
    value_columns=["amt"],                   # numeric columns to extract from
    mode=FeatureExtractionMode.COMPREHENSIVE,
    impute_missing=True
)

print(f"\nTSFRESH RESULTS:")
print(f"  Features extracted: {len(ts_result.feature_names)}")
print(f"  Extraction time: {ts_result.extraction_time:.2f}s")
print(f"  Extraction mode: {ts_result.extraction_mode.value}")
if ts_result.warnings:
    print(f"  Warnings: {ts_result.warnings}")

In [0]:
# Preview extracted time-series features
print(f"TSFresh feature names ({len(ts_result.feature_names)}):")
for name in ts_result.feature_names[:15]:
    print(f"  - {name}")

ts_result.feature_matrix.head()

In [0]:
# Filter to only statistically relevant features
import pandas as pd

# Build per-entity target: majority label per cc_num
target_per_entity = (
    df.groupBy("cc_num")
    .agg({"is_fraud": "max"})
    .toPandas()
    .set_index("cc_num")["max(is_fraud)"]
)

# Align indices
common_idx = ts_result.feature_matrix.index.intersection(target_per_entity.index)
target_aligned = target_per_entity.loc[common_idx]

ts_filtered = ts_engine.filter_relevant_features(
    result=ts_result,
    target=target_aligned,
    fdr_level=0.05
)

print(f"Relevant features: {len(ts_filtered.relevant_features or [])} / {len(ts_result.feature_names)}")
if ts_filtered.relevant_features:
    for name in ts_filtered.relevant_features[:10]:
        print(f"  - {name}")

# Merge tsfresh features back into the main DataFrame (entity-level -> transaction-level join)
# Use filtered features if available, otherwise fall back to all extracted features
ts_to_merge = ts_filtered if ts_filtered.relevant_features else ts_result
df = ts_engine.to_spark(ts_to_merge, df, id_column="cc_num")

print(f"\nDataFrame after tsfresh merge: {len(df.columns)} columns")

---
## 7. Auto Feature Generation + Preprocessing

At this point `df` contains the original columns **plus** featuretools DFS
features and tsfresh time-series features merged in the previous steps.

Now we generate additional features (interactions, binning, datetime
extraction) using AutoFeatureGenerator and encode everything through the
Spark ML pipeline for XGBoost training.

In [0]:
from backend.core.features.auto_feature_generator import AutoFeatureGenerator

# Re-create schema checker on the enriched DataFrame (includes featuretools + tsfresh columns)
schema_checker = SchemaChecks(dataframe=df, problem=problem)
schema_checker.check()

feature_gen = AutoFeatureGenerator(
    schema_checks=schema_checker,
    problem=problem
)

numerical_cols = schema_checker.get_typed_col(col_type="numerical")
categorical_cols = schema_checker.get_typed_col(col_type="categorical")
datetime_cols = schema_checker.get_typed_col(col_type="datetime")

# Remove target from feature lists
for col_list in [numerical_cols, categorical_cols, datetime_cols]:
    if problem.target in col_list:
        col_list.remove(problem.target)

print(f"Enriched DataFrame before auto-generation: {len(df.columns)} columns")
print(f"  Numerical: {len(numerical_cols)}, Categorical: {len(categorical_cols)}, Datetime: {len(datetime_cols)}")

df_with_features = feature_gen.generate_all_features(
    include_numerical=True,
    include_interactions=True,
    include_binning=True,
    include_datetime=True,
    include_string=False,
    numerical_columns=numerical_cols,
    categorical_columns=categorical_cols,
    datetime_columns=datetime_cols
)

print(f"\nFEATURE GENERATION SUMMARY:")
print(f"  Input features (original + featuretools + tsfresh): {len(df.columns)}")
print(f"  Total features after auto-generation: {len(df_with_features.columns)}")
print(f"  New features generated: {len(df_with_features.columns) - len(df.columns)}")

In [0]:
from backend.core.features.process import PreProcessVariables
from backend.core.discovery import SchemaChecks
from pyspark.sql import functions as F

# Restore original method to avoid recursion from previous runs
schema_checker.get_typed_col = SchemaChecks.get_typed_col.__get__(schema_checker, SchemaChecks)

# Filter categorical columns by cardinality (keep < 20 unique values)
categorical_cols = schema_checker.get_typed_col(col_type="categorical")
if problem.target in categorical_cols:
    categorical_cols.remove(problem.target)

print("Filtering categorical columns by cardinality...")
low_cardinality_cols = []
for col in categorical_cols:
    distinct_count = df_with_features.select(F.countDistinct(col)).collect()[0][0]
    if distinct_count < 20:
        low_cardinality_cols.append(col)
        print(f"  + {col}: {distinct_count} unique values")
    else:
        print(f"  - {col}: {distinct_count} unique values (excluded)")

# Keep only first 15 numerical features to fit under model size limits
all_numerical_cols = schema_checker.get_typed_col(col_type="numerical")
if problem.target in all_numerical_cols:
    all_numerical_cols.remove(problem.target)
reduced_numerical_cols = all_numerical_cols[:15]

print(f"\nUsing {len(low_cardinality_cols)} categorical + {len(reduced_numerical_cols)} numerical columns")

# Reduce dataframe to selected columns
cols_to_keep = [problem.target] + low_cardinality_cols + reduced_numerical_cols + [problem.date_column]
cols_to_keep = [c for c in cols_to_keep if c in df_with_features.columns]
df_reduced = df_with_features.select(*cols_to_keep)

# Sample for faster training
df_sample = df_reduced.sample(fraction=0.00025, seed=42)
print(f"Sample size: {df_sample.count():,} rows")

# Patch schema_checker to use filtered columns
original_get_typed_col = SchemaChecks.get_typed_col
def patched_get_typed_col(self, col_type):
    if col_type == "categorical":
        return low_cardinality_cols
    elif col_type == "numerical":
        return reduced_numerical_cols
    return original_get_typed_col(self, col_type)

schema_checker.get_typed_col = patched_get_typed_col.__get__(schema_checker, SchemaChecks)

# Build Spark ML pipeline: StringIndexer -> OneHotEncoder -> VectorAssembler
pre_process_variables = PreProcessVariables(
    dataframe=df_with_features,
    problem=problem,
    schema_checks=schema_checker,
    train_dataframe=df_sample
)

transformed_df, feature_names, feature_output_col, feature_map = pre_process_variables.process()

# Restore original method
schema_checker.get_typed_col = original_get_typed_col.__get__(schema_checker, SchemaChecks)

print(f"\nPREPROCESSING SUMMARY:")
print(f"  Encoded features: {len(feature_names)}")
print(f"  Feature vector column: {feature_output_col}")
print(f"  Total columns after preprocessing: {len(transformed_df.columns)}")

---
## 8. XGBoost - Model Training

Train a SparkXGBoost classifier and evaluate on train/test sets.

In [0]:
from backend.core.features.feature_selector import FeatureSelector

feature_selector = FeatureSelector(
    problem=problem,
    transformed_df=transformed_df,
    feature_names=feature_names,
    feature_col=feature_output_col,
    feature_idx_name_mapping=feature_map,
    train_split=0.8
)

print("Training XGBoost model...")
feature_selector.train_model()
print("XGBoost training complete!")

In [0]:
# Evaluate on train and test sets
print("XGBOOST EVALUATION:")
print("=" * 40)

print("\nTrain Set:")
train_metrics = feature_selector.evaluate(train=True)

print("\nTest Set:")
test_metrics = feature_selector.evaluate(train=False)

In [0]:
# Feature importance from XGBoost
print("TOP 20 FEATURES BY IMPORTANCE:")
print("-" * 40)

importance_list = feature_selector.get_feature_importances()
for i, (feature, importance) in enumerate(importance_list[:20]):
    print(f"  {i+1:2d}. {feature}: {importance:.4f}")

---
## 9. SHAP - Model Explainability

Use SHAP (SHapley Additive exPlanations) to understand feature contributions:
- **Summary plot**: Distribution of SHAP values per feature
- **Bar plot**: Mean absolute SHAP values (global importance)
- **Waterfall plot**: Single-prediction explanation
- **Feature value impacts**: Aggregated tree split analysis

In [0]:
# Global SHAP analysis
print("SHAP ANALYSIS:")
print("=" * 50)

shap_results = feature_selector.get_shap_analysis(
    sample_size=1000,
    plot=True,
    plot_type='all'  # generates both summary and bar plots
)

print("\nSHAP Feature Importance (Top 15):")
print("-" * 40)
print(shap_results['feature_importance'].head(15).to_string(index=False))

In [0]:
# Display SHAP summary plot
from IPython.display import Image, display
display(Image(filename='shap_summary_plot.png'))

In [0]:
# Display SHAP bar plot
display(Image(filename='shap_bar_plot.png'))

In [0]:
# Feature value impacts from tree splits
print("FEATURE VALUE IMPACTS (Aggregated from Tree Splits):")
print("-" * 60)

feature_impacts = feature_selector.get_feature_value_impacts(top_n=15)
print(feature_impacts.to_string(index=False))

In [0]:
# Explain a single prediction
print("EXPLAINING A SINGLE PREDICTION:")
print("=" * 50)

explanation = feature_selector.explain_prediction(instance_idx=0, use_test=True)

print(f"\nInstance Index: {explanation['instance_idx']}")
print(f"Prediction: {explanation['prediction']}")
print(f"Probability: {explanation['probability']}")
print(f"Actual Label: {explanation['actual_label']}")
print(f"Base Value: {explanation['base_value']:.4f}")

print("\nTop 5 Positive Contributors (pushing toward positive class):")
print(explanation['top_positive'][['Feature', 'Value', 'SHAP_Value']].to_string(index=False))

print("\nTop 5 Negative Contributors (pushing toward negative class):")
print(explanation['top_negative'][['Feature', 'Value', 'SHAP_Value']].to_string(index=False))

In [0]:
# Waterfall plot for single prediction
feature_selector.plot_shap_waterfall(instance_idx=0, use_test=True)
display(Image(filename='shap_waterfall_instance_0.png'))

---
## 10. Insight Analyzer - Microsegments & Lift vs Support

SparkBeyond-style feature discovery that identifies:
- **Lift**: How much better a feature condition performs vs. baseline (e.g., x3.09 = 3x better)
- **Support**: Percentage of data covered by the condition
- **RIG**: Relative Information Gain - information value about the target
- **Microsegments**: Powerful combinations of feature conditions

In [0]:
from backend.core.features.insight_analyzer import FeatureInsightAnalyzer

print("FEATURE INSIGHT ANALYSIS")
print("=" * 60)

insight_analyzer = FeatureInsightAnalyzer(
    df=df_with_features,      # use original data (not transformed)
    problem=problem,
    schema_checks=schema_checker,
    n_bins=10,                 # bins for numeric features
    min_support=0.01,          # minimum 1% support
    min_lift=1.1               # minimum 10% lift over baseline
)

result = insight_analyzer.get_analysis_result(discover_microsegments=True)

print(f"\nAnalysis Summary:")
print(f"  Target Class: {result.target_class}")
print(f"  Baseline Rate: {result.baseline_rate*100:.2f}%")
print(f"  Total Records: {result.total_count:,}")
print(f"  Total Insights Found: {result.summary['total_insights']}")
print(f"  Microsegments Found: {result.summary['total_microsegments']}")

In [0]:
# Top insights sorted by lift
print("TOP 20 FEATURE INSIGHTS (Sorted by Lift):")
print("-" * 70)

insights_df = insight_analyzer.to_dataframe(top_n=20)
display_cols = ['Condition', 'Lift', 'Support', 'Support_Count', 'RIG', 'Class_Rate']
print(insights_df[display_cols].to_string(index=False))

In [0]:
# Display microsegments (feature combinations)
print("TOP MICROSEGMENTS (Feature Combinations):")
print("-" * 70)

if result.microsegments:
    for i, micro in enumerate(result.microsegments[:10], 1):
        print(f"\n{i}. {micro.name}")
        print(f"   Lift: x{micro.lift:.2f} | Support: {micro.support*100:.1f}% ({micro.support_count:,}) | RIG: {micro.rig:.3f}")
        print(f"   Class Rate: {micro.class_rate*100:.1f}% vs Baseline: {micro.baseline_rate*100:.1f}%")
else:
    print("No microsegments found that improve over individual features.")

In [0]:
# Lift vs Support scatter plot (SparkBeyond-style)
print("Generating Lift vs Support Scatter Plot...")
fig = insight_analyzer.plot_lift_support_scatter(
    top_n=50,
    highlight_microsegments=True,
    save_path='insight_lift_support.png'
)

display(Image(filename='insight_lift_support.png'))

In [0]:
# Top insights bar chart
print("Generating Top Insights Bar Chart...")
fig = insight_analyzer.plot_top_insights(
    top_n=15,
    metric='lift',
    save_path='insight_top_features.png'
)

display(Image(filename='insight_top_features.png'))

In [0]:
# Full insights table for exploration
print("FULL INSIGHTS TABLE:")
display(insight_analyzer.display_insights_table(top_n=30))

---
## Summary

### Cumulative Feature Pipeline

```
df (raw) ──► featuretools DFS ──► tsfresh time-series ──► AutoFeatureGenerator ──► Spark ML Pipeline ──► XGBoost / SHAP / Insights
              (merge back)         (merge back)            (interactions, bins)     (encode + vectorize)
```

| Step | Library | What it adds |
|------|---------|--------------|
| 3 | **ydata-profiling** | Data exploration, alerts & recommendations |
| 5 | **featuretools** | DFS transform features (add, subtract, multiply numeric) |
| 6 | **tsfresh** | Entity-level time-series statistics (mean, std, min, max, ...) |
| 7 | **AutoFeatureGenerator** | Interactions, binning, datetime extraction on enriched df |
| 8 | **XGBoost** | SparkXGBoost training with feature importance |
| 9 | **SHAP** | Shapley value explanations (summary, bar, waterfall) |
| 10 | **Insight Analyzer** | Lift, Support, RIG analysis & microsegment discovery |

### Key Metrics from Insight Analysis:
- **Lift**: How much better a feature condition performs vs. baseline
- **Support**: Percentage of data covered by the condition
- **RIG**: Relative Information Gain - how much information the feature provides about the target

In [0]:
# Cleanup
insight_analyzer.cleanup()
print("Demo complete!")